# Solution vs whole completion, judge vs generation context (Countdown)

Fills the grid of `tab_context_countdown` and every row of `tab_judges_countdown`. The GRPO run `mean-34` / `-cont100b` dumped 128 training completions every
20 steps (steps 0-380, 2,560 in all, 160 puzzle groups of 16). Each is scored by per-token log-probs in seven views:
- **judge_sft**: the SFT policy (GRPO's start) in the judge format: neutral system prompt + bare puzzle, then
  solution + "Answer: ..." + EOS (the GRPO reward's text and filter: solution and answer present, solution <= 128 tokens)
- **judge_base**: the same with the adapter off (Llama-3.2-3B-Instruct)
- **judge_solver**, **judge_corrupt**: the same under the two DPO judges (adapters continued from the SFT policy):
  solver-made errors (Consistency_Countdown) and corrupted reasoning (Consistency_Corrupt_Judge)
- **gen_sol**: the SFT policy on its own GRPO prompt and completion, scoring only the tokens of the solution and answer
  contents (same filter as the judge views)
- **gen_full**: the same forward pass, scoring the whole completion (+ EOS if it finished)
- **gen_full_base**: the whole completion after the same policy prompt, under the base model (adapter off)

Scores per view: `mean`, `sum`, `low10` (sum of the 10 lowest token log-probs), `min` (lowest), token count. The
[checks] cell compares the means with the earlier scorings (Judge Score Probe, Corrupt Judge): they must agree.
Inference only, no vLLM; a few minutes on an A100 (40 GB is enough). Views already in `context_scores.csv` are loaded,
not scored again (RESCORE = True scores all).

Run all: [config] → [drive + HF] → [install] (no restart) → [setup] → [data] → [score] → [checks] → [release]
(releases the runtime with UNASSIGN_WHEN_DONE).

In [ ]:
# [config] The run whose dumps are scored, the SFT policy and the two DPO judges that score them.
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
SFT_RUN = "Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000"
DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
ADAPTER_PATH = f"{DRIVE}/countdown/sft/{SFT_RUN}/adapter"                      # the SFT policy (GRPO's start)
SOLVER_DPO_RUN = "judge_dpo_swap30extra30slip20miss20_n2250_n3x2250_b0.1_lr2e-05_ep1"               # Consistency_Countdown
CORRUPT_DPO_RUN = "judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1"   # Corrupt_Judge
JUDGE_ADAPTERS = {"solver": f"{DRIVE}/countdown/sft/{SFT_RUN}/{SOLVER_DPO_RUN}/adapter",         # judge_solver view
                  "corrupt": f"{DRIVE}/countdown/sft/{SFT_RUN}/{CORRUPT_DPO_RUN}/adapter"}       # judge_corrupt view
GRPO_DIR = f"{DRIVE}/countdown/grpo/{SFT_RUN}-judge_dpo-start_sft"
RUN = f"{GRPO_DIR}/runs/grpo-mean-missmed-mix3_0.5-expr-kl0.02-swap30extra30slip20miss20_n2250_n3x2250-mean-34"
DUMP_DIRS = [(RUN, 0, 99), (RUN + "-cont100b", 100, 10**6)]   # (run, first step, last step), as the score probe
OUT_DIR = f"{RUN}-cont100b/context_score"
PLAUSIBILITY_MAX_SOLUTION_TOKENS = 128   # judge and gen_sol views: as the GRPO reward (longer, or no answer: not scored)
LOW_K = 10
CHUNK = 4                                # sequences per forward pass
RESCORE = False                          # [score]: reuse OUT_DIR/context_scores.csv if it is there
# earlier scorings of the same completions, for [checks] (skipped if missing)
CHECKS = {"probe": f"{RUN}-cont100b/judge_score_probe/scores.csv",                   # full_mean (SFT policy)
          "zero_shot": f"{RUN}-cont100b/judge_score_probe/zero_shot_scores.csv",     # zs_llama3b_mean (base)
          "corrupt": f"{DRIVE}/countdown/sft/{SFT_RUN}/{CORRUPT_DPO_RUN}/dump_scores.csv"}   # sft, base, solver, corrupt
UNASSIGN_WHEN_DONE = True                # release the Colab runtime after the last cell (stops the compute bill)

In [ ]:
# [drive + HF] Mount Drive and log in to HF (Colab Secret HF_TOKEN; Llama is gated). Re-running is harmless.
import os
from google.colab import drive, userdata
from huggingface_hub import login
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
login(userdata.get("HF_TOKEN"))

In [ ]:
# [install] PEFT for the adapters (no vLLM, no TRL: inference only), on Colab's torch and transformers; no restart
# (nothing it replaces is imported yet). Colab's torchao is too old for peft (peft refuses to import): removed.
import importlib.util, subprocess, sys
if importlib.util.find_spec("peft"):
    print("[install] skipped: peft is installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft"], check=True)
    print("[install] done")
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

In [ ]:
# [setup] The Countdown code (Drive), the tokenizer, the prompt helpers.
import os, sys, re, glob, json, importlib, math
import numpy as np
import pandas as pd
from fractions import Fraction
from transformers import AutoTokenizer
for f in (f"{ADAPTER_PATH}/adapter_model.safetensors", *(f"{p}/adapter_model.safetensors" for p in JUDGE_ADAPTERS.values()),
          f"{CODE_DIR}/rewards.py", f"{CODE_DIR}/traces.py"):
    assert os.path.exists(f), f"missing {f}"
sys.path.insert(0, CODE_DIR)
import countdown, traces, rewards as R
for _m in (countdown, traces, R):
    importlib.reload(_m)
from traces import countdown_prompt
from rewards import tag, bare_question

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = "<|finetune_right_pad_id|>"   # not <|eot_id|> (the EOS)
tokenizer.chat_template = re.sub(r'strftime_now\([^)]*\)', '"26 Jul 2024"', tokenizer.chat_template)  # pinned date

def chat_prompt_ids(tok, messages) -> list[int]:
    """Templated prompt as token ids, no extra BOS (as the GRPO notebook)."""
    return tok(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
               add_special_tokens=False)["input_ids"]

NEUTRAL_SYSTEM_PROMPT = "Please solve the following problem step-by-step, making sure that each step is well-justified."

In [ ]:
# [data] The dumped completions: step, puzzle, numbers (in prompt order), target, the checker's verdict, the logged
# judge mean, the response text; and the wrong answer's kind from its <solution> (a false step = "fake"; all steps
# true = "true steps, wrong numbers"; no <answer> = "no answer").
EQ = re.compile(r"(-?\d+(?:\.\d+)?)\s*([+\-*/x×])\s*(-?\d+(?:\.\d+)?)\s*=\s*(-?\d+(?:\.\d+)?)")

def _true(a, op, b, c):
    a, b, c = Fraction(a), Fraction(b), Fraction(c)
    try:
        return {"+": a + b, "-": a - b, "*": a * b, "x": a * b, "×": a * b, "/": a / b}[op] == c
    except ZeroDivisionError:
        return False

def kind(correct, text):
    if correct:
        return "correct"
    ans, sol = tag(text, "answer"), tag(text, "solution") or ""
    if not ans:
        return "no answer"
    return "fake" if any(not _true(*e) for e in EQ.findall(sol)) else "true steps, wrong numbers"

rows = []
for run, lo, hi in DUMP_DIRS:
    for f in sorted(glob.glob(f"{run}/step_*.txt")):
        step = int(re.search(r"step_(\d+)", f).group(1))
        if not lo <= step <= hi:
            continue
        for blk in open(f).read().split("### Sample ")[1:]:
            sc = re.search(r"puzzle=(\d+) correct=(\w+) judge_mean=(\S+)", blk)
            prompt = blk.split("[Prompt]\n", 1)[1].split("\n\n[Response]\n", 1)[0]
            resp = blk.split("[Response]\n", 1)[1].split("\n\n" + "=" * 20, 1)[0]
            pm = re.match(r"Using the numbers (.*?), create an expression that equals (\d+)\.", prompt)
            nums, tgt = [int(x) for x in pm.group(1).split(", ")], int(pm.group(2))
            assert countdown_prompt(nums, tgt)[-1]["content"] == prompt, f"prompt differs: {f}"
            rows.append({"step": step, "pid": int(sc.group(1)), "n": len(nums), "numbers": nums, "target": tgt,
                         "correct": sc.group(2) == "True", "judge_logged": float(sc.group(3)), "text": resp,
                         "kind": kind(sc.group(2) == "True", resp)})
data = pd.DataFrame(rows)
data["group"] = data.step.astype(str) + "/" + data.pid.astype(str)
assert (data.groupby("group").size() == 16).all(), "a group without 16 completions"
print(f"{len(data)} completions, {data.group.nunique()} groups, steps {sorted(data.step.unique())}")
print(data.kind.value_counts().to_string())

In [ ]:
# [score] One base model, the SFT adapter on (judge_sft, gen_sol, gen_full) or off (judge_base, gen_full_base), or a DPO
# judge's adapter (judge_solver, judge_corrupt: the judge format, as judge_sft). Per view
# the per-token log-probs of the scored tokens -> mean, sum, low10, min, tokens. gen_sol and gen_full come from the same
# forward pass (the completion after the policy's prompt): gen_sol keeps the tokens overlapping the solution and answer
# contents (the last <solution> / <answer> blocks, whitespace trimmed; no tags, no EOS). gen_full_base: the same
# sequence under the base model. Cached in OUT_DIR/context_scores.csv; views missing from the cache are scored and
# merged in (RESCORE = True scores every view again).
import torch
SCORES = ("mean", "sum", "low10", "min", "tokens")
VIEWS = ("judge_sft", "judge_base", "judge_solver", "judge_corrupt", "gen_sol", "gen_full", "gen_full_base")
ADAPTERS = ("sft", *JUDGE_ADAPTERS, None)   # scoring order; None = adapter off
_cache = f"{OUT_DIR}/context_scores.csv"

def summarize(lp):
    lp = np.asarray(lp, dtype=np.float64)
    return {"mean": lp.mean(), "sum": lp.sum(), "low10": np.sort(lp)[:LOW_K].sum(), "min": lp.min(), "tokens": len(lp)}

def content_span(text, name):
    """(start, end) of the last <name> block's content, whitespace trimmed (as rewards.tag), or None."""
    m = list(re.finditer(rf"<{name}>(.*?)</{name}>", text, re.DOTALL))
    if not m:
        return None
    s, e = m[-1].span(1)
    while s < e and text[s].isspace():
        s += 1
    while e > s and text[e - 1].isspace():
        e -= 1
    return (s, e) if e > s else None

def items_for(i, r):
    """(i, adapter, prompt ids, scored ids, {view: mask over scored ids}) for one completion."""
    out = []
    sol, ans = tag(r.text, "solution"), tag(r.text, "answer")
    eligible = bool(sol and ans) and len(tokenizer(sol, add_special_tokens=False)["input_ids"]) <= PLAUSIBILITY_MAX_SOLUTION_TOKENS
    if eligible:
        p = chat_prompt_ids(tokenizer, [{"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
                                        {"role": "user", "content": bare_question(countdown_prompt(r.numbers, r.target))}])
        t = tokenizer(f"{sol}\nAnswer: {ans}", add_special_tokens=False)["input_ids"] + [tokenizer.eos_token_id]
        out.append((i, "sft", p, t, {"judge_sft": [True] * len(t)}))
        out.append((i, None, p, t, {"judge_base": [True] * len(t)}))
        out += [(i, a, p, t, {f"judge_{a}": [True] * len(t)}) for a in JUDGE_ADAPTERS]
    enc = tokenizer(r.text, add_special_tokens=False, return_offsets_mapping=True)
    t, offs = list(enc["input_ids"]), list(enc["offset_mapping"])
    if r.text.rstrip().endswith("</answer>"):
        t, offs = t + [tokenizer.eos_token_id], offs + [(len(r.text), len(r.text))]
    masks = {"gen_full": [True] * len(t)}
    if eligible:
        spans = [x for x in (content_span(r.text, "solution"), content_span(r.text, "answer")) if x]
        masks["gen_sol"] = [any(a < se and b > ss for ss, se in spans) and b > a for a, b in offs]
    gp = chat_prompt_ids(tokenizer, countdown_prompt(r.numbers, r.target))
    out.append((i, "sft", gp, t, masks))
    out.append((i, None, gp, t, {"gen_full_base": [True] * len(t)}))
    return out

todo = list(VIEWS)
if os.path.exists(_cache) and not RESCORE:
    _c = pd.read_csv(_cache)
    assert len(_c) == len(data) and (_c[["step", "pid", "correct"]].values == data[["step", "pid", "correct"]].values).all(), \
        "context_scores.csv is for other completions: set RESCORE = True"
    todo = [v for v in VIEWS if f"{v}_mean" not in _c.columns]
    for _col in [c for c in _c.columns if c.startswith(VIEWS)]:
        data[_col] = _c[_col].values
    print(f"[score] loaded {_cache}; to score: {todo or 'nothing'} (RESCORE = True scores every view again)")
if todo:
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="cuda"),
                                      ADAPTER_PATH, adapter_name="sft").eval()
    for _name, _path in JUDGE_ADAPTERS.items():
        if f"judge_{_name}" in todo:
            model.load_adapter(_path, adapter_name=_name)
    items = [(j, a, p, t, {v: m for v, m in masks.items() if v in todo})
             for i, r in enumerate(data.itertuples()) for j, a, p, t, masks in items_for(i, r)]
    items = [x for x in items if x[4]]
    items.sort(key=lambda x: (ADAPTERS.index(x[1]), len(x[2]) + len(x[3])))    # per adapter; similar lengths per chunk
    chunks = [c for a in ADAPTERS for ai in [[x for x in items if x[1] == a]] for c in
              (ai[s:s + CHUNK] for s in range(0, len(ai), CHUNK))]
    scores = {}
    with torch.no_grad():
        for s, chunk in enumerate(chunks):
            L = max(len(p) + len(t) for _, _, p, t, _ in chunk)
            ids = torch.full((len(chunk), L), tokenizer.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, (_, _, p, t, _) in enumerate(chunk):
                ids[j, :len(p) + len(t)] = torch.tensor(p + t)
                attn[j, :len(p) + len(t)] = 1
            if chunk[0][1] is None:
                with model.disable_adapter():
                    logits = model(input_ids=ids.cuda(), attention_mask=attn.cuda()).logits
            else:
                model.set_adapter(chunk[0][1])
                logits = model(input_ids=ids.cuda(), attention_mask=attn.cuda()).logits
            for j, (i, _, p, t, masks) in enumerate(chunk):
                z = logits[j, len(p) - 1:len(p) + len(t) - 1].float()          # positions predicting t
                lp = (z.gather(-1, torch.tensor(t, device=z.device)[:, None])[:, 0] - torch.logsumexp(z, -1)).cpu().numpy()
                for v, m in masks.items():
                    if any(m):
                        scores[(i, v)] = summarize(lp[np.array(m)])
            del logits
            if s % 200 == 0:
                print(f"{s}/{len(chunks)} chunks")
    for v in todo:
        for m in SCORES:
            data[f"{v}_{m}"] = [scores.get((i, v), {}).get(m, np.nan) for i in range(len(data))]
    os.makedirs(OUT_DIR, exist_ok=True)
    data.drop(columns=["text"]).to_csv(_cache, index=False)
    print("saved", _cache)
print(data[[f"{v}_tokens" for v in VIEWS]].describe().round(1).to_string())

In [ ]:
# [checks] (1) The means agree with the earlier scorings of the same completions (same text, model and context):
# judge_sft, judge_base, judge_solver, judge_corrupt vs the Corrupt Judge notebook's [dump probe] ("sft", "base", "solver",
# "corrupt"), judge_base also vs the probe's zero-shot llama3b mean,
# gen_full vs the probe's full_mean. Small differences come from bf16 and package versions (median |diff| < 0.02 or < 1% of the value). (2) Win rate of correct over wrong
# per view and score (same puzzle group, ties 1/2) on the rows every view scored.
def _median_diff(a, b):
    d = (pd.Series(a) - pd.Series(b)).abs().dropna()
    return d.median(), d.max(), len(d)

for name, path in CHECKS.items():
    if not os.path.exists(path):
        print(f"[checks] {name}: {path} missing, skipped")
        continue
    old = pd.read_csv(path)
    assert len(old) == len(data) and (old.step.values == data.step.values).all() and (old.pid.values == data.pid.values).all()
    pairs = {"probe": [("gen_full_mean", "full_mean"), ("gen_full_tokens", "full_tokens")],
             "zero_shot": [("judge_base_mean", "zs_llama3b_mean"), ("judge_base_low10", "zs_llama3b_low10")],
             "corrupt": [("judge_sft_mean", "sft"), ("judge_base_mean", "base"), ("judge_solver_mean", "solver"),
                         ("judge_corrupt_mean", "corrupt")]}[name]
    if name == "probe":
        pairs.append(("gen_full_base_tokens", "full_tokens"))
    for new, ref in pairs:
        med, mx, n = _median_diff(data[new], old[ref])
        rel = med / old[ref].abs().median()     # low10 is ~30x the mean in size; SFT means are near 0: either suffices
        print(f"[checks] {new} vs {name}.{ref}: median |diff| {med:.4f} ({rel:.2%} of the median value), "
              f"max {mx:.4f} over {n}")
        assert (med == 0) if ref.endswith("tokens") else (med < 0.02 or rel < 0.01), f"{new} does not reproduce {name}.{ref}"

def win(df, col):
    w = t = 0
    for _, g in df.dropna(subset=[col]).groupby("group"):
        diff = g.loc[g.correct, col].values[:, None] - g.loc[~g.correct, col].values[None, :]
        w, t = w + (diff > 0).sum() + 0.5 * (diff == 0).sum(), t + diff.size
    return w / t if t else float("nan")

common = data.dropna(subset=[f"{v}_mean" for v in VIEWS])
res = pd.DataFrame({v: {m: win(common, f"{v}_{m}") for m in ("mean", "sum", "low10", "min")} for v in VIEWS}).T
print(f"\nwin rate, correct over wrong ({len(common)} completions scored in every view, "
      f"{common.groupby('group').correct.apply(lambda x: 0 < x.mean() < 1).sum()} mixed groups):")
print(res.round(3).to_string())

In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before
# this one, and the runtime stays up for a look).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()